# Złożony workflow – zadania

Zaimplementuj pipeline do **anonimizacji danych osobowych w tekście**. Pipeline powinien:
1. Równolegle wykryć język tekstu i sprawdzić czy zawiera dane osobowe **(równoległe node'y)**
2. Na podstawie wyników zdecydować, czy tekst wymaga anonimizacji **(conditional routing)**
3. Jeśli tak – LLM anonimizuje dane i ponownie sprawdza czy coś zostało pominięte **(cykl)**
4. Zbudować końcowy raport z przetwarzania

## Stan (TypedDict)

1. Zdefiniuj `State` zawierający:
- `text: str` – przetwarzany tekst (może być nadpisany przez node anonimizujący)
- `language: str` – wykryty język
- `pii: dict` – wynik analizy danych osobowych, np. `{"has_pii": bool, "found_items": list[str]}`
- `anonymize_attempts: int` – ile razy próbowano anonimizować tekst
- `is_safe: bool` – czy tekst jest bezpieczny do udostępnienia
- `report: str` – końcowy raport z przetwarzania

---
(czas: 5 min.)

In [ ]:
# ...

## Funkcje routingu

2. Napisz funkcję routingu `next_after_evaluate(state: State, config: RunnableConfig) -> str`, która:
- pobiera `max_attempts` z `configurable` (domyślnie `2`)
- jeśli `pii["has_pii"]` jest `True` i `anonymize_attempts < max_attempts`: zwraca `"anonymize"`
- w przeciwnym razie: zwraca `"build_report"`

---
(czas: 6 min.)

In [ ]:
# ...

## Node'y

3. Napisz node `start_parallel_node(state: State)`, który nie modyfikuje stanu i służy wyłącznie jako punkt startowy dla równoległych gałęzi. Zwróć pusty słownik `{}`.

---
(czas: 2 min.)

In [ ]:
# ...

4. Napisz node `detect_language_node(state: State, config: RunnableConfig)`, który za pomocą LLM wykrywa język tekstu.

Użyj modelu Pydantic z jednym polem `language: str` i metody `with_structured_output`. Zapisz wykryty język (np. `"polski"`, `"angielski"`) do pola `language`.

---
(czas: 10 min.)

In [ ]:
# ...

5. Napisz node `check_pii_node(state: State, config: RunnableConfig)`, który za pomocą LLM sprawdza czy tekst zawiera dane osobowe (imiona i nazwiska, adresy e-mail, numery telefonów, adresy, numery PESEL itp.).

Użyj modelu Pydantic z polami `has_pii: bool` i `found_items: list[str]` oraz metody `with_structured_output`. Zapisz wynik jako słownik do pola `pii`.

Ten node jest wywoływany **dwukrotnie** w grafie: raz równolegle z `detect_language`, a następnie ponownie po każdej anonimizacji.

---
(czas: 10 min.)

In [ ]:
# ...

6. Napisz node `evaluate_node(state: State)`, który ustawia `is_safe` na `True` jeśli `pii["has_pii"]` wynosi `False`, w przeciwnym razie na `False`.

---
(czas: 3 min.)

In [ ]:
# ...

7. Napisz node `anonymize_node(state: State, config: RunnableConfig)`, który:
- wysyła do LLM polecenie zastąpienia wszystkich danych osobowych odpowiednimi placeholderami (np. `[IMIĘ]`, `[NAZWISKO]`, `[EMAIL]`, `[TELEFON]`, `[ADRES]`)
- nadpisuje `text` zanonimizowaną wersją
- zwiększa `anonymize_attempts` o 1

---
(czas: 8 min.)

In [ ]:
# ...

8. Napisz node `build_report_node(state: State)`, który formatuje końcowy raport zawierający: przetworzony tekst, wykryty język, listę znalezionych danych osobowych, liczbę prób anonimizacji i status (`bezpieczny` / `wymaga weryfikacji`).

---
(czas: 5 min.)

In [ ]:
# ...

## Graf i uruchomienie

9. Zbuduj `StateGraph` z następującą strukturą:
- `START → start_parallel`
- `start_parallel → detect_language` i `start_parallel → check_pii` (dwie osobne krawędzie – równoległe)
- `detect_language → evaluate` i `check_pii → evaluate`
- `evaluate → conditional (next_after_evaluate) → anonymize` lub `build_report`
- `anonymize → check_pii` (zamknięcie cyklu)
- `build_report → END`

Skompiluj pipeline i wyświetl wizualizację.

---
(czas: 12 min.)

In [ ]:
# ...

10. Uruchom pipeline na trzech różnych tekstach i wypisz `report` z każdego uruchomienia:
- Tekst **bez danych osobowych** (krótka ścieżka bez anonimizacji)
- Tekst **z danymi osobowymi** – np. fragment zgłoszenia z imieniem, emailem i numerem telefonu (widoczny cykl)
- Ten sam tekst z `max_attempts=1` – sprawdź czy po jednej rundzie anonimizacji wszystkie dane zostały usunięte

---
(czas: 8 min.)

In [ ]:
# ...